In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from lightgbm import LGBMClassifier

In [ ]:
df = pd.read_csv("../data/processed/cleaned_telco.csv")

df.head()

In [ ]:
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

print("Input shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
X = pd.get_dummies(X, drop_first=True, dtype=int)

X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1
)

model.fit(X_train, y_train)

print("Model trained successfully!")

In [ ]:
y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]

print("Predictions:", y_pred[:10])
print("Churn probabilities:", y_prob[:10])

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("LightGBM Confusion Matrix")
plt.show()

In [ ]:
joblib.dump(model, "../models/churn_model.pkl")

joblib.dump(
    X_train.columns.tolist(),
    "../models/feature_columns.pkl"
)

print("Model saved successfully!")

In [ ]:
loaded_model = joblib.load("../models/churn_model.pkl")

print(loaded_model.predict(X_test.head()))

In [ ]:
import shap

In [ ]:
explainer = shap.TreeExplainer(model)

shap_values = explainer(X_test)

print("SHAP values calculated successfully!")

In [ ]:
shap.plots.bar(shap_values, max_display=10)

In [ ]:
shap.plots.beeswarm(shap_values, max_display=10)

In [ ]:
shap.plots.waterfall(shap_values[0])

In [ ]:
shap.plots.bar(
    shap_values,
    max_display=10,
    show=False
)

plt.savefig(
    "../reports/shap_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()